# Asian Voting Behavior Diagnostics — 10 X 5 STV

Diagnostics for Asian voter/candidate representation dynamics in the `10 X 5 STV` run (`configs/10x5-stv.json`). This notebook:

1. Bootstraps a DataFrame of election outcomes across the entire simulation.
2. Finds the 5 sampled districts with the highest Asian VAP share.
3. For each of those districts, tallies how many ballots ranked the Asian-slate candidate(s) 1st, 2nd, and 3rd — pooled across every voter model and replicate.


In [1]:
import json
import tempfile
import zipfile
from pathlib import Path

import pandas as pd
from votekit import RankProfile


In [2]:
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN_NAME = "10 X 5 STV"
DISTRICT_COUNT = 10

CONFIG_PATH = REPO_ROOT / "configs" / "10x5-stv.json"
OUTPUTS_DIR = REPO_ROOT / "outputs" / RUN_NAME

with open(CONFIG_PATH) as f:
    config = json.load(f)

MODES = config["voter_models"]
NUM_REPS = config["num_reps"]
MODES, NUM_REPS


(['slate_pl', 'slate_bt'], 10)

## 1. Election outcomes across the entire simulation

One row per (plan, district, replicate, voter model, election method), including `focal_seats` — seats won by Asian-preferred ("A"-slate) candidates — plus the citywide baselines. This is the same table `summarize_results` writes to CSV.


In [3]:
summary_csv = OUTPUTS_DIR / "summaries" / f"{RUN_NAME}_summary.csv"
outcomes_df = pd.read_csv(summary_csv)
print(outcomes_df.shape)
outcomes_df.head()


(10000, 18)


,run_name,plan,num_districts,seats_per_district,election_method,mode,district_id,rep,simulation_index,focal_group,focal_seats,total_vap_20,asian_nhpi_vap_20,combined_support,seats_W,seats_B,seats_H,seats_A
0,10 X 5 STV,0,10,5,FASTSTV,slate_bt,0,0,192,A,0,203883.0,585.0,0.153914,0,4,0,0
1,10 X 5 STV,0,10,5,FASTSTV,slate_bt,1,0,168,A,0,243334.0,33317.0,0.153914,3,1,0,0
2,10 X 5 STV,0,10,5,FASTSTV,slate_bt,2,0,323,A,0,214482.0,1234.0,0.153914,1,0,2,0
3,10 X 5 STV,0,10,5,FASTSTV,slate_bt,3,0,283,A,0,217833.0,20127.0,0.153914,0,5,0,0
4,10 X 5 STV,0,10,5,FASTSTV,slate_bt,4,0,487,A,0,210849.0,29214.0,0.153914,3,0,1,0


## 2. Top 5 sampled districts by Asian population share

Pulled from the cross-run `district_demographics.csv` export (raw VAP facts, independent of any run's turnout/cohesion settings), filtered to the 10-district ensemble this run uses. District identity isn't comparable across different sampled plans, so each row here is one concrete (plan, district) instance.


In [4]:
demographics_csv = REPO_ROOT / "outputs" / "cross_run_summaries" / "district_demographics.csv"
demographics_df = pd.read_csv(demographics_csv)

ensemble_df = demographics_df[demographics_df["district_count"] == DISTRICT_COUNT]

top5_districts = ensemble_df.sort_values(
    "asian_nhpi_vap_20_share", ascending=False
).head(5)
top5_districts[["plan_idx", "district_id", "asian_nhpi_vap_20_share", "total_vap"]]


,plan_idx,district_id,asian_nhpi_vap_20_share,total_vap
169,3200,9,0.228993,246767.0
304,6000,4,0.223461,246396.0
256,5000,6,0.220429,234148.0
188,3600,8,0.219849,238013.0
155,3000,5,0.219844,216499.0


## 3. First/second/third-place vote counts for the Asian candidate(s)

For each of the 5 districts above: candidate slates are apportioned per-district (see `pipeline/settings_generator.py`), so we first check which candidate(s), if any, belong to the "A" slate in that specific district's settings file. Then, pooling across every voter model and replicate profile for that (plan, district), we count how many ballots ranked that candidate 1st, 2nd, or 3rd (by ballot weight).

"First/second/third place" here means ballot ranking position (how voters ordered their preferences), not STV tabulation-round vote totals after transfers.


In [5]:
def settings_path_for(plan_idx: int, district_id: int) -> Path:
    return (
        OUTPUTS_DIR / "settings" / str(DISTRICT_COUNT)
        / f"{RUN_NAME}_{DISTRICT_COUNT}_sample_settings_district_plan_"
          f"{plan_idx:03d}_district_{district_id:02d}.json"
    )


def profile_members_for(plan_idx: int, district_id: int) -> list[str]:
    members = []
    for mode in MODES:
        for rep in range(NUM_REPS):
            members.append(
                f"{mode}/{DISTRICT_COUNT}/{RUN_NAME}_{DISTRICT_COUNT}_profile_district_plan_"
                f"{plan_idx:03d}_district_{district_id:02d}_v{rep}.csv"
            )
    return members


def load_profile_from_zip(zip_path: Path, member_name: str) -> RankProfile:
    with zipfile.ZipFile(zip_path) as archive:
        data = archive.read(member_name)
    with tempfile.NamedTemporaryFile(mode="wb", suffix=".csv", delete=False) as tmp:
        tmp.write(data)
        temp_path = Path(tmp.name)
    try:
        return RankProfile.from_csv(temp_path)
    finally:
        temp_path.unlink()


def rank_vote_counts_for_candidate(profile: RankProfile, candidate: str, max_rank: int = 3):
    """[1st-place votes, 2nd-place votes, 3rd-place votes] for one candidate in one profile."""
    counts = [0.0] * max_rank
    for ballot in profile.ballots:
        for i in range(min(max_rank, len(ballot.ranking))):
            if candidate in ballot.ranking[i]:
                counts[i] += ballot.weight
                break  # a candidate appears in at most one ranking position per ballot
    return counts


In [6]:
zip_path = OUTPUTS_DIR / "profiles.zip"

records = []
for _, row in top5_districts.iterrows():
    plan_idx = int(row["plan_idx"])
    district_id = int(row["district_id"])

    settings = json.loads(settings_path_for(plan_idx, district_id).read_text())
    asian_candidates = settings["slate_to_candidates"].get("A", [])

    if not asian_candidates:
        records.append({
            "plan_idx": plan_idx,
            "district_id": district_id,
            "asian_vap_share": row["asian_nhpi_vap_20_share"],
            "candidate": None,
            "first_place_votes": 0.0,
            "second_place_votes": 0.0,
            "third_place_votes": 0.0,
        })
        continue

    members = profile_members_for(plan_idx, district_id)

    for candidate in asian_candidates:
        totals = [0.0, 0.0, 0.0]
        for member in members:
            profile = load_profile_from_zip(zip_path, member)
            counts = rank_vote_counts_for_candidate(profile, candidate)
            totals = [t + c for t, c in zip(totals, counts)]

        records.append({
            "plan_idx": plan_idx,
            "district_id": district_id,
            "asian_vap_share": row["asian_nhpi_vap_20_share"],
            "candidate": candidate,
            "first_place_votes": totals[0],
            "second_place_votes": totals[1],
            "third_place_votes": totals[2],
        })

vote_counts_df = pd.DataFrame(records)
vote_counts_df


,plan_idx,district_id,asian_vap_share,candidate,first_place_votes,second_place_votes,third_place_votes
0,3200,9,0.228993,A1,6537.0,7707.0,10057.0
1,3200,9,0.228993,A2,8771.0,9881.0,11877.0
2,3200,9,0.228993,A3,8443.0,9929.0,11708.0
3,6000,4,0.223461,A1,5316.0,5976.0,7143.0
4,6000,4,0.223461,A2,5057.0,5789.0,8140.0
5,6000,4,0.223461,A3,5672.0,6102.0,7996.0
6,6000,4,0.223461,A4,6416.0,6994.0,8371.0
7,5000,6,0.220429,A1,11488.0,13150.0,14291.0
8,5000,6,0.220429,A2,10588.0,12416.0,14214.0
9,3600,8,0.219849,A1,12279.0,13318.0,13766.0


In [9]:
vote_counts_df['total_votes'] = vote_counts_df[['first_place_votes','second_place_votes','third_place_votes']].sum(axis=1)
vote_counts_df

,plan_idx,district_id,asian_vap_share,candidate,first_place_votes,second_place_votes,third_place_votes,total_votes
0,3200,9,0.228993,A1,6537.0,7707.0,10057.0,24301.0
1,3200,9,0.228993,A2,8771.0,9881.0,11877.0,30529.0
2,3200,9,0.228993,A3,8443.0,9929.0,11708.0,30080.0
3,6000,4,0.223461,A1,5316.0,5976.0,7143.0,18435.0
4,6000,4,0.223461,A2,5057.0,5789.0,8140.0,18986.0
5,6000,4,0.223461,A3,5672.0,6102.0,7996.0,19770.0
6,6000,4,0.223461,A4,6416.0,6994.0,8371.0,21781.0
7,5000,6,0.220429,A1,11488.0,13150.0,14291.0,38929.0
8,5000,6,0.220429,A2,10588.0,12416.0,14214.0,37218.0
9,3600,8,0.219849,A1,12279.0,13318.0,13766.0,39363.0


## 4. Are these votes coming from Asian voters specifically, or all voters?

Short answer: we can't tell from the ballots tallied above, for two separate reasons.

1. **The saved profiles don't retain per-ballot voter-bloc identity.** `profile.to_csv()` was called
   with its default `include_voter_set=False`, and even in memory, votekit's merged bloc-slate
   generators (`slate_pl_profile_generator` / `slate_bt_profile_generator`) don't tag which bloc cast
   each ballot before merging — so bloc identity isn't recoverable from the ballots we already
   tallied in section 3.
2. **This run's voter-bloc structure merges Asian and White voters together.** `10 X 5 STV` uses the
   "standard 3-bloc" structure (`configs/10x5-stv.json`'s `"blocs"` key: `{"W-A": ["W", "A"], "B": [...],
   "H": [...]}`), so Asian voters' ballots are never generated separately here — the finest granularity
   available is `W-A` vs. `B` vs. `H`, not "Asian voters" on their own. (The `Asian Bloc Separate -
   Basic` run models Asian voters as their own bloc instead; that's the run to use for true
   Asian-voter-only attribution.)

What we *can* do: regenerate ballots **per bloc** for each of the top-5 districts using votekit's
`slate_pl_profiles_by_bloc_generator` / `slate_bt_profiles_by_bloc_generator` (these return one
`RankProfile` per bloc instead of one pre-merged profile), and tally first/second/third-place support
for the Asian candidate(s) separately by bloc. This is a **fresh, independent random draw** from the
same district settings — it won't reproduce the exact ballots tallied in section 3 — but it shows the
same underlying pattern: how much of a candidate's support comes from their own (`W-A`) bloc versus
crossover from `B`/`H`.


In [ ]:
from votekit.ballot_generator import (
    BlocSlateConfig,
    slate_pl_profiles_by_bloc_generator,
    slate_bt_profiles_by_bloc_generator,
)

by_bloc_generator = {
    "slate_pl": slate_pl_profiles_by_bloc_generator,
    "slate_bt": slate_bt_profiles_by_bloc_generator,
}


def build_config_for_settings(settings: dict) -> BlocSlateConfig:
    bloc_config = BlocSlateConfig(
        n_voters=settings["num_voters"],
        slate_to_candidates=settings["slate_to_candidates"],
        bloc_proportions=settings["bloc_proportions"],
        cohesion_mapping=settings["cohesion_parameters"],
    )
    bloc_config.set_dirichlet_alphas(settings["alphas"])
    return bloc_config


In [ ]:
bloc_records = []
for _, row in top5_districts.iterrows():
    plan_idx = int(row["plan_idx"])
    district_id = int(row["district_id"])

    settings = json.loads(settings_path_for(plan_idx, district_id).read_text())
    asian_candidates = settings["slate_to_candidates"].get("A", [])
    if not asian_candidates:
        continue

    bloc_config = build_config_for_settings(settings)

    for mode in MODES:
        profiles_by_bloc = by_bloc_generator[mode](bloc_config)

        for candidate in asian_candidates:
            for bloc, profile in profiles_by_bloc.items():
                counts = rank_vote_counts_for_candidate(profile, candidate)
                bloc_records.append({
                    "plan_idx": plan_idx,
                    "district_id": district_id,
                    "mode": mode,
                    "candidate": candidate,
                    "bloc": bloc,
                    "first_place_votes": counts[0],
                    "second_place_votes": counts[1],
                    "third_place_votes": counts[2],
                })

bloc_vote_counts_df = pd.DataFrame(bloc_records)
bloc_vote_counts_df


### First-place vote share by bloc

Normalizing within each (district, candidate, mode) group shows what fraction of a candidate's
first-place support comes from their own `W-A` bloc versus crossover from `B`/`H` — the direct
answer to "is this just [Asian-inclusive] `W-A` voters, or broader support?"


In [ ]:
group_cols = ["plan_idx", "district_id", "candidate", "mode"]
bloc_vote_counts_df["first_place_share"] = bloc_vote_counts_df.groupby(group_cols)[
    "first_place_votes"
].transform(lambda s: s / s.sum())

bloc_vote_counts_df[group_cols + ["bloc", "first_place_votes", "first_place_share"]]
